# GDP per Capita Relative to the World Average

This notebook documents a World Data Atlas visualization project.

The goal is to create an animated world map showing how each country's GDP per capita compares with the global average over time.

**Period:** 1960–2024  
**Source:** World Bank  
**Indicator:** `NY.GDP.PCAP.CD` — GDP per capita, current US$  
**Output:** PNG frames and MP4 animation


## Project idea

Raw GDP per capita values are hard to compare visually because the distribution is extremely skewed.

Instead of mapping absolute GDP per capita, this project compares each country to the world average:

\[
\text{relative GDP per capita} = \frac{\text{country GDP per capita}}{\text{world GDP per capita}}
\]

To make differences below and above the world average more readable, the final mapped value uses a log transformation:

\[
\text{mapped value} = \log\left(\frac{\text{country GDP per capita}}{\text{world GDP per capita}}\right)
\]

This makes the scale more balanced:

- `0` = equal to the world average
- negative values = below the world average
- positive values = above the world average


In [ ]:
from pathlib import Path
import pandas as pd
import sqlalchemy as sa

# Project paths
BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "results"
OUTPUT_DIR.mkdir(exist_ok=True)

# Import local settings
# settings.py should contain the SQL Server connection string.
import sys
etl_path = (BASE_DIR.parent / "ETL").resolve()
sys.path.insert(0, str(etl_path))

import settings as s

engine = sa.create_engine(s.connection_string)


## SQL query

The query returns one row per country and year.

The `value` column is not raw GDP per capita. It is the log of the country-to-world GDP per capita ratio.

The world average is taken from the same World Bank table, where `country_name = 'World'`.


In [ ]:
YEAR = 2005

query = f"""
SELECT 
    D.[country_code],
    D.[country_name],
    D.[country_id],
    LOG(CAST(D.[value] AS FLOAT) / CAST(W.[value] AS FLOAT)) AS [value]
FROM [World_Data_Atlas].[worldbank].[data] AS D
LEFT JOIN (
    SELECT 
        [wb_id],
        [iso2_code],
        [name],
        [is_country]
    FROM [World_Data_Atlas].[worldbank].[entities]
    WHERE [is_country] = 1
) AS ENT 
    ON ENT.[name] = D.[country_name]
INNER JOIN (
    SELECT 
        [year],
        [value]
    FROM [World_Data_Atlas].[worldbank].[data]
    WHERE 
        [indicator_code] = 'NY.GDP.PCAP.CD'
        AND [country_name] = 'World'
        AND [value] IS NOT NULL
        AND [value] > 0
) AS W
    ON W.[year] = D.[year]
WHERE 
    D.[indicator_code] = 'NY.GDP.PCAP.CD'
    AND D.[year] = {YEAR}
    AND ENT.[is_country] = 1
    AND D.[value] IS NOT NULL
    AND D.[value] > 0
ORDER BY [value] DESC;
"""

df = pd.read_sql(query, engine)
df.head()


## Prepare map input

The reusable map function expects a dictionary:

```python
{
    "USA": 1.72,
    "CHN": -0.21,
    ...
}
```

where the key is a country ISO code and the value is the metric to visualize.


In [ ]:
highlight_countries = df.set_index("country_code")["value"].to_dict()

len(highlight_countries), list(highlight_countries.items())[:5]


## Legend design

The map uses log values internally, but the legend should be readable for humans.

For example:

- `log(1) = 0` → `1×`
- `log(2.7) ≈ 1` → `2.7×`
- `log(0.37) ≈ -1` → `0.37×`

The final animation uses a fixed scale from `-3` to `2.5`.

This corresponds roughly to:

- `<0.05×` the world average
- `1×` the world average
- `>12×` the world average


In [ ]:
legend_min_value = -3
legend_max_value = 2.5

legend_ticks = [-3, -2, -1, 0, 1, 2, 2.5]

legend_tick_labels = [
    "<0.05×",
    "0.14×",
    "0.37×",
    "1×",
    "2.7×",
    "7.4×",
    ">12×"
]


## Create one map frame

The visualization is generated using a reusable `create_world_map()` function.

The map is styled as a dark world map with a fixed legend scale, making all years comparable across the animation.


In [ ]:
# Import reusable map function
module_path = (BASE_DIR.parent / "modules").resolve()
sys.path.insert(0, str(module_path))

from graphic_map import create_world_map

create_world_map(
    highlight_countries=highlight_countries,
    map_title="GDP per Capita Relative to World Average",
    map_subtitle=f"{YEAR} · World Bank data · log scale",
    source_text="Source: World Bank",
    legend_title="GDP pc vs world avg",
    legend_min_value=legend_min_value,
    legend_max_value=legend_max_value,
    legend_ticks=legend_ticks,
    legend_tick_labels=legend_tick_labels,
    save_image=True,
    output_file=str(OUTPUT_DIR / f"gdp_relative_world_{YEAR}.png"),
    show_plot=True
)


## Generate frames for all years

The same logic can be wrapped in a loop to export one PNG frame per year.


In [ ]:
START_YEAR = 1960
END_YEAR = 2024

for year in range(START_YEAR, END_YEAR + 1):
    query = f"""
    SELECT 
        D.[country_code],
        D.[country_name],
        D.[country_id],
        LOG(CAST(D.[value] AS FLOAT) / CAST(W.[value] AS FLOAT)) AS [value]
    FROM [World_Data_Atlas].[worldbank].[data] AS D
    LEFT JOIN (
        SELECT 
            [wb_id],
            [iso2_code],
            [name],
            [is_country]
        FROM [World_Data_Atlas].[worldbank].[entities]
        WHERE [is_country] = 1
    ) AS ENT 
        ON ENT.[name] = D.[country_name]
    INNER JOIN (
        SELECT 
            [year],
            [value]
        FROM [World_Data_Atlas].[worldbank].[data]
        WHERE 
            [indicator_code] = 'NY.GDP.PCAP.CD'
            AND [country_name] = 'World'
            AND [value] IS NOT NULL
            AND [value] > 0
    ) AS W
        ON W.[year] = D.[year]
    WHERE 
        D.[indicator_code] = 'NY.GDP.PCAP.CD'
        AND D.[year] = {year}
        AND ENT.[is_country] = 1
        AND D.[value] IS NOT NULL
        AND D.[value] > 0;
    """

    df_year = pd.read_sql(query, engine)
    highlight_countries = df_year.set_index("country_code")["value"].to_dict()

    create_world_map(
        highlight_countries=highlight_countries,
        map_title="GDP per Capita Relative to World Average",
        map_subtitle=f"{year} · World Bank data · log scale",
        source_text="Source: World Bank",
        legend_title="GDP pc vs world avg",
        legend_min_value=legend_min_value,
        legend_max_value=legend_max_value,
        legend_ticks=legend_ticks,
        legend_tick_labels=legend_tick_labels,
        save_image=True,
        output_file=str(OUTPUT_DIR / f"gdp_relative_world_{year}.png"),
        show_plot=False
    )

print("Frames exported:", len(list(OUTPUT_DIR.glob("*.png"))))


## Create MP4 animation

The exported PNG frames are combined into a video using MoviePy.

Lower `fps` makes the animation longer and slower.  
For example:

- `fps=10` → faster
- `fps=8` → slightly slower
- `fps=6` → noticeably slower


In [ ]:
from moviepy import ImageSequenceClip

images = sorted(str(p) for p in OUTPUT_DIR.glob("*.png"))

output_file = BASE_DIR / "gdp_per_capita_relative_to_world_average.mp4"

clip = ImageSequenceClip(images, fps=8)

clip.write_videofile(
    str(output_file),
    codec="libx264",
    audio=False,
    preset="medium",
    bitrate="5000k",
    ffmpeg_params=[
        "-vf", "scale=1920:-2",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart"
    ]
)

print("Done:", output_file)


## Notes and limitations

- Early years have fewer available country observations.
- Current Natural Earth country borders are used, so historical entities such as the Soviet Union, Czechoslovakia, or Yugoslavia are not represented as historical borders.
- Countries without available World Bank data are shown as no-data countries.
- The metric is relative to the World Bank `World` aggregate, not a manually calculated country average.
- The visualization is intended for public data storytelling, not as a full economic model.


## Main insight

The animation highlights long-term changes in global relative prosperity.

Some countries converged toward or above the world average, while others remained far below it. The log scale makes both high-income outliers and low-income differences visible in the same map.
